In [1]:
# ============================================================
# Notebook 08 — Professor Code Review Fixes
# Fix 1: One-Hot Encoding for diagnosis + Target Encoding
# Fix 2: Remove PCA from LightGBM
# Fix 3: 5-Fold Stratified Cross Validation
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold,
                                     cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, recall_score,
                             precision_score, f1_score,
                             roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 123

print("All libraries loaded!")

All libraries loaded!


In [2]:
# Load the merged dataset with SDOH
# We go back to this because we need to redo encoding properly
df = pd.read_csv('../data/processed/diabetic_with_sdoh.csv')

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Readmission rate: {df['readmitted_30'].mean()*100:.1f}%")
print(f"\nColumns:")
print(df.columns.tolist())

Rows:    101,766
Columns: 48
Readmission rate: 11.2%

Columns:
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_30', 'BPHIGH', 'CHECKUP', 'CHOLSCREEN', 'DEPRESSION', 'DIABETES', 'OBESITY']


In [3]:
# ============================================================
# FIX 1: PROPER ENCODING
# Problem: Label Encoding created fake ordering
# Circulatory=0, Diabetes=1, Digestive=2 — model thinks
# Digestive > Diabetes > Circulatory which is WRONG
#
# Solution:
# - One-Hot Encoding for diag_1, diag_2, diag_3 (9 categories)
# - Target Encoding for medical_specialty (73 categories)
# - Label Encoding kept for binary/ordinal columns only
# ============================================================

df_encoded = df.copy()

# Step 1 — Check current diagnosis values
print("Current diagnosis categories:")
print(df_encoded['diag_1'].value_counts())
print(f"\nMedical specialty unique values: {df_encoded['medical_specialty'].nunique()}")

Current diagnosis categories:
diag_1
0    30437
7    18172
8    14423
2     9475
1     8757
4     6974
3     5117
5     4957
6     3433
9       21
Name: count, dtype: int64

Medical specialty unique values: 73


In [4]:
# Step 2 — Map numbers back to diagnosis category names
diag_map = {
    0: 'Circulatory', 1: 'Diabetes', 2: 'Digestive',
    3: 'Genitourinary', 4: 'Injury', 5: 'Musculoskeletal',
    6: 'Neoplasms', 7: 'Other', 8: 'Respiratory', 9: 'Unknown'
}

df_encoded['diag_1'] = df_encoded['diag_1'].map(diag_map)
df_encoded['diag_2'] = df_encoded['diag_2'].map(diag_map)
df_encoded['diag_3'] = df_encoded['diag_3'].map(diag_map)

print("Diagnosis categories restored:")
print(df_encoded['diag_1'].value_counts())

# Step 3 — One-Hot Encode diagnosis columns
# drop_first=True removes one category to avoid multicollinearity
df_encoded = pd.get_dummies(
    df_encoded,
    columns=['diag_1', 'diag_2', 'diag_3'],
    drop_first=True,
    dtype=int
)

print(f"\nOne-Hot Encoding applied!")
print(f"New diagnosis columns created:")
diag_cols = [c for c in df_encoded.columns if c.startswith('diag_')]
print(diag_cols)
print(f"\nDataset shape after One-Hot: {df_encoded.shape}")

Diagnosis categories restored:
diag_1
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown               21
Name: count, dtype: int64

One-Hot Encoding applied!
New diagnosis columns created:
['diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Genitourinary', 'diag_1_Injury', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Other', 'diag_1_Respiratory', 'diag_1_Unknown', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Genitourinary', 'diag_2_Injury', 'diag_2_Musculoskeletal', 'diag_2_Neoplasms', 'diag_2_Other', 'diag_2_Respiratory', 'diag_2_Unknown', 'diag_3_Diabetes', 'diag_3_Digestive', 'diag_3_Genitourinary', 'diag_3_Injury', 'diag_3_Musculoskeletal', 'diag_3_Neoplasms', 'diag_3_Other', 'diag_3_Respiratory', 'diag_3_Unknown']

Dataset shape after One-Hot: (101766, 72)


In [8]:
# Step 4 — Split data FIRST before target encoding
X_temp = df_encoded.drop(columns=['readmitted_30'])
y_temp = df_encoded['readmitted_30']

# 60/30/10 split
X_train, X_temp2, y_train, y_temp2 = train_test_split(
    X_temp, y_temp,
    test_size=0.40,
    random_state=RANDOM_SEED,
    stratify=y_temp
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp2, y_temp2,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_temp2
)

print(f"Split sizes:")
print(f"  Training:   {len(X_train):,} rows (60%)")
print(f"  Validation: {len(X_val):,} rows (30%)")
print(f"  Test:       {len(X_test):,} rows (10%) SEALED")

# Step 5 — Target Encoding for medical_specialty
# Calculate from TRAINING DATA ONLY
print(f"\nApplying Target Encoding for medical_specialty...")

y_train_named = y_train.rename('readmitted_30')
specialty_means = (X_train
                   .join(y_train_named)
                   .groupby('medical_specialty')['readmitted_30']
                   .mean())

overall_mean = y_train.mean()

# Apply to all splits in one line each
X_train['specialty_risk'] = X_train['medical_specialty'].map(specialty_means).fillna(overall_mean)
X_val['specialty_risk']   = X_val['medical_specialty'].map(specialty_means).fillna(overall_mean)
X_test['specialty_risk']  = X_test['medical_specialty'].map(specialty_means).fillna(overall_mean)

# Drop original medical_specialty column
X_train = X_train.drop(columns=['medical_specialty'])
X_val   = X_val.drop(columns=['medical_specialty'])
X_test  = X_test.drop(columns=['medical_specialty'])

print(f"\nTop 10 specialties by readmission rate:")
print(specialty_means.sort_values(ascending=False).head(10))

print(f"\nTarget Encoding applied!")
print(f"specialty_risk mean:  {X_train['specialty_risk'].mean():.4f}")
print(f"specialty_risk range: {X_train['specialty_risk'].min():.4f} to {X_train['specialty_risk'].max():.4f}")
print(f"\nFinal feature count: {X_train.shape[1]}")

Split sizes:
  Training:   61,059 rows (60%)
  Validation: 30,530 rows (30%)
  Test:       10,177 rows (10%) SEALED

Applying Target Encoding for medical_specialty...

Top 10 specialties by readmission rate:
medical_specialty
0     0.600000
49    0.333333
14    0.304348
38    0.250000
17    0.227273
15    0.221374
66    0.181818
25    0.167464
1     0.166667
19    0.154786
Name: readmitted_30, dtype: float64

Target Encoding applied!
specialty_risk mean:  0.1116
specialty_risk range: 0.0000 to 0.6000

Final feature count: 71


In [9]:
# Calculate scientific weights from correlation
correlations = df_encoded.corr()['readmitted_30'].abs()
features_for_weight = ['num_medications', 'num_procedures', 'number_diagnoses']
total_corr = sum(correlations[f] for f in features_for_weight)
weights = {f: correlations[f] / total_corr for f in features_for_weight}

w_meds = weights['num_medications']
w_proc = weights['num_procedures']
w_diag = weights['number_diagnoses']

print(f"Scientific weights:")
print(f"  num_medications:  {w_meds:.4f}")
print(f"  num_procedures:   {w_proc:.4f}")
print(f"  number_diagnoses: {w_diag:.4f}")

# Apply feature engineering to all 3 splits
for X in [X_train, X_val, X_test]:
    # Medication features
    diabetes_meds = ['metformin', 'repaglinide', 'glimepiride',
                     'glipizide', 'glyburide', 'pioglitazone',
                     'rosiglitazone', 'insulin', 'change']
    available_meds = [m for m in diabetes_meds if m in X.columns]
    X['total_diabetes_meds']  = (X[available_meds] > 0).sum(axis=1)
    X['med_complexity_score'] = (X['num_medications'] * w_meds +
                                  X['num_procedures']  * w_proc +
                                  X['number_diagnoses'] * w_diag)
    X['high_med_burden']      = (X['num_medications'] >= X['num_medications'].quantile(0.75)).astype(int)
    X['on_insulin']           = (X['insulin'] > 0).astype(int)
    X['multiple_med_changes'] = (X['change'] > 0).astype(int)

    # Clinical risk features
    X['prior_utilization']     = (X['number_inpatient'] * 0.5 +
                                   X['number_emergency'] * 0.3 +
                                   X['number_outpatient'] * 0.2)
    X['high_prior_inpatient']  = (X['number_inpatient'] >= 2).astype(int)
    X['emergency_admission']   = (X['admission_source_id'] == 7).astype(int)
    X['long_stay']             = (X['time_in_hospital'] >= X['time_in_hospital'].quantile(0.75)).astype(int)
    X['high_diagnosis_burden'] = (X['number_diagnoses'] >= 7).astype(int)

    # Interaction features
    X['inpatient_x_meds']        = X['number_inpatient'] * X['num_medications']
    X['utilization_x_diagnoses'] = X['number_inpatient'] * X['number_diagnoses']
    X['age_x_diagnoses']         = X['age'] * X['number_diagnoses']
    X['emergency_x_inpatient']   = X['number_emergency'] * X['number_inpatient']
    X['meds_x_stay']             = X['num_medications'] * X['time_in_hospital']
    X['obesity_x_diabetes']      = X['OBESITY'] * X['DIABETES']
    X['composite_risk']          = (X['high_prior_inpatient'] +
                                    X['high_diagnosis_burden'] +
                                    X['high_med_burden'] +
                                    X['long_stay'] +
                                    X['emergency_admission'])

# Remove near zero variance from training
variance = X_train.select_dtypes(include='number').var()
low_var = variance[variance < 0.01].index.tolist()
X_train = X_train.drop(columns=low_var, errors='ignore')
X_val   = X_val.drop(columns=low_var, errors='ignore')
X_test  = X_test.drop(columns=low_var, errors='ignore')

print(f"\nFeature engineering complete!")
print(f"Final feature count: {X_train.shape[1]}")

Scientific weights:
  num_medications:  0.3836
  num_procedures:   0.1220
  number_diagnoses: 0.4943

Feature engineering complete!
Final feature count: 72


In [10]:
# Apply SMOTE to training set only
print("Applying SMOTE to training set only...")

smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"SMOTE applied!")
print(f"\nBefore SMOTE:")
print(f"  Class 0: {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)")
print(f"  Class 1: {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)")
print(f"\nAfter SMOTE:")
print(f"  Class 0: {(y_train_sm==0).sum():,} (50.0%)")
print(f"  Class 1: {(y_train_sm==1).sum():,} (50.0%)")
print(f"  New size: {len(X_train_sm):,} rows")

Applying SMOTE to training set only...
SMOTE applied!

Before SMOTE:
  Class 0: 54,245 (88.8%)
  Class 1: 6,814 (11.2%)

After SMOTE:
  Class 0: 54,245 (50.0%)
  Class 1: 54,245 (50.0%)
  New size: 108,490 rows


In [13]:
# ============================================================
# FIX 2: PCA ONLY FOR LOGISTIC REGRESSION
# Professor said: tree models like XGBoost and LightGBM
# work by asking "Is feature > threshold?" — PCA destroys
# this interpretability. Only use PCA for linear models.
# ============================================================

print("Applying PCA for Logistic Regression only")

# Scale first (required for both PCA and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Apply PCA — keep 95% variance
pca = PCA(n_components=0.95, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"PCA ready for Logistic Regression!")
print(f"  Original features: {X_train_sm.shape[1]}")
print(f"  PCA components:    {X_train_pca.shape[1]}")
print(f"  Variance kept:     {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"\nXGBoost and LightGBM will use X_train_sm directly")
print(f"Logistic Regression will use X_train_pca")

Applying PCA for Logistic Regression only
PCA ready for Logistic Regression!
  Original features: 72
  PCA components:    50
  Variance kept:     95.7%

XGBoost and LightGBM will use X_train_sm directly
Logistic Regression will use X_train_pca


In [15]:
# ============================================================
# FIX 3: 5-FOLD STRATIFIED CROSS VALIDATION
# Professor said: single train/val split is unreliable
# 5-fold CV trains 5 times on different portions
# giving more reliable average score
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Model 1 — Logistic Regression WITH PCA
print("Training Logistic Regression with PCA + 5-Fold CV...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    class_weight='balanced'
)

lr_cv_scores = cross_val_score(
    lr_model, X_train_pca, y_train_sm,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f"Logistic Regression + PCA done!")
print(f"\n5-Fold CV ROC-AUC scores:")
for i, score in enumerate(lr_cv_scores):
    print(f"  Fold {i+1}: {score:.4f}")
print(f"\nMean ROC-AUC: {lr_cv_scores.mean():.4f}")
print(f"Std:          {lr_cv_scores.std():.4f}")
print(f"Range:        {lr_cv_scores.min():.4f} — {lr_cv_scores.max():.4f}")

# Train final LR on full training data for validation evaluation
lr_model.fit(X_train_pca, y_train_sm)
y_val_prob_lr = lr_model.predict_proba(X_val_pca)[:, 1]

fpr, tpr, thresholds = roc_curve(y_val, y_val_prob_lr)
opt_thresh_lr = thresholds[np.argmax(tpr - fpr)]
y_val_pred_lr = (y_val_prob_lr >= opt_thresh_lr).astype(int)

auc_lr  = roc_auc_score(y_val, y_val_prob_lr)
rec_lr  = recall_score(y_val, y_val_pred_lr)
prec_lr = precision_score(y_val, y_val_pred_lr)
f1_lr   = f1_score(y_val, y_val_pred_lr)

print(f"\nValidation Results (threshold={opt_thresh_lr:.4f}):")
print(f"  ROC-AUC:   {auc_lr:.4f}")
print(f"  Recall:    {rec_lr:.4f}")
print(f"  Precision: {prec_lr:.4f}")
print(f"  F1:        {f1_lr:.4f}")

Training Logistic Regression with PCA + 5-Fold CV...
Logistic Regression + PCA done!

5-Fold CV ROC-AUC scores:
  Fold 1: 0.9223
  Fold 2: 0.9198
  Fold 3: 0.9188
  Fold 4: 0.9208
  Fold 5: 0.9224

Mean ROC-AUC: 0.9208
Std:          0.0014
Range:        0.9188 — 0.9224

Validation Results (threshold=0.1959):
  ROC-AUC:   0.5516
  Recall:    0.4514
  Precision: 0.1344
  F1:        0.2072


In [17]:
# Model 2 — XGBoost WITHOUT PCA (professor suggestion)
print("Training XGBoost WITHOUT PCA + 5-Fold CV...")

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    scale_pos_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=RANDOM_SEED,
    eval_metric='logloss',
    verbosity=0
)

# 5-fold CV on original features (NO PCA)
xgb_cv_scores = cross_val_score(
    xgb_model, X_train_sm, y_train_sm,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f"XGBoost done!")
print(f"\n5-Fold CV ROC-AUC scores:")
for i, score in enumerate(xgb_cv_scores):
    print(f"  Fold {i+1}: {score:.4f}")
print(f"\nMean ROC-AUC: {xgb_cv_scores.mean():.4f}")
print(f"Std:          {xgb_cv_scores.std():.4f}")
print(f"Range:        {xgb_cv_scores.min():.4f} — {xgb_cv_scores.max():.4f}")

# Train final model for validation evaluation
xgb_model.fit(X_train_sm, y_train_sm)
y_val_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]

fpr, tpr, thresholds = roc_curve(y_val, y_val_prob_xgb)
opt_thresh_xgb = thresholds[np.argmax(tpr - fpr)]
y_val_pred_xgb = (y_val_prob_xgb >= opt_thresh_xgb).astype(int)

auc_xgb  = roc_auc_score(y_val, y_val_prob_xgb)
rec_xgb  = recall_score(y_val, y_val_pred_xgb)
prec_xgb = precision_score(y_val, y_val_pred_xgb)
f1_xgb   = f1_score(y_val, y_val_pred_xgb)

print(f"\nValidation Results (threshold={opt_thresh_xgb:.4f}):")
print(f"  ROC-AUC:   {auc_xgb:.4f}")
print(f"  Recall:    {rec_xgb:.4f}")
print(f"  Precision: {prec_xgb:.4f}")
print(f"  F1:        {f1_xgb:.4f}")

Training XGBoost WITHOUT PCA + 5-Fold CV...
XGBoost done!

5-Fold CV ROC-AUC scores:
  Fold 1: 0.9561
  Fold 2: 0.9548
  Fold 3: 0.9542
  Fold 4: 0.9536
  Fold 5: 0.9563

Mean ROC-AUC: 0.9550
Std:          0.0011
Range:        0.9536 — 0.9563

Validation Results (threshold=0.3476):
  ROC-AUC:   0.6143
  Recall:    0.6313
  Precision: 0.1467
  F1:        0.2380


In [18]:
# Model 3 — LightGBM WITHOUT PCA (professor suggestion)
print("Training LightGBM WITHOUT PCA + 5-Fold CV...")

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    scale_pos_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_SEED,
    verbose=-1
)

# 5-fold CV on original features (NO PCA)
lgb_cv_scores = cross_val_score(
    lgb_model, X_train_sm, y_train_sm,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f"LightGBM done!")
print(f"\n5-Fold CV ROC-AUC scores:")
for i, score in enumerate(lgb_cv_scores):
    print(f"  Fold {i+1}: {score:.4f}")
print(f"\nMean ROC-AUC: {lgb_cv_scores.mean():.4f}")
print(f"Std:          {lgb_cv_scores.std():.4f}")
print(f"Range:        {lgb_cv_scores.min():.4f} — {lgb_cv_scores.max():.4f}")

# Train final model for validation evaluation
lgb_model.fit(X_train_sm, y_train_sm)
y_val_prob_lgb = lgb_model.predict_proba(X_val)[:, 1]

fpr, tpr, thresholds = roc_curve(y_val, y_val_prob_lgb)
opt_thresh_lgb = thresholds[np.argmax(tpr - fpr)]
y_val_pred_lgb = (y_val_prob_lgb >= opt_thresh_lgb).astype(int)

auc_lgb  = roc_auc_score(y_val, y_val_prob_lgb)
rec_lgb  = recall_score(y_val, y_val_pred_lgb)
prec_lgb = precision_score(y_val, y_val_pred_lgb)
f1_lgb   = f1_score(y_val, y_val_pred_lgb)

print(f"\nValidation Results (threshold={opt_thresh_lgb:.4f}):")
print(f"  ROC-AUC:   {auc_lgb:.4f}")
print(f"  Recall:    {rec_lgb:.4f}")
print(f"  Precision: {prec_lgb:.4f}")
print(f"  F1:        {f1_lgb:.4f}")

Training LightGBM WITHOUT PCA + 5-Fold CV...
LightGBM done!

5-Fold CV ROC-AUC scores:
  Fold 1: 0.9564
  Fold 2: 0.9530
  Fold 3: 0.9527
  Fold 4: 0.9535
  Fold 5: 0.9566

Mean ROC-AUC: 0.9544
Std:          0.0017
Range:        0.9527 — 0.9566

Validation Results (threshold=0.4811):
  ROC-AUC:   0.6270
  Recall:    0.5826
  Precision: 0.1546
  F1:        0.2444


In [20]:
# Final comparison of all approaches
results = pd.DataFrame({
    'Model': [
        'XGBoost original (Week 5)',
        'XGBoost + interactions (Week 6)',
        'LR + PCA (Fix 2)',
        'XGBoost no PCA + One-Hot (Fix 1+2+3)',
        'LightGBM no PCA + One-Hot (Fix 1+2+3)'
    ],
    'CV_ROC_AUC': [
        'N/A', 'N/A',
        f"{lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}",
        f"{xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}",
        f"{lgb_cv_scores.mean():.4f} ± {lgb_cv_scores.std():.4f}"
    ],
    'Val_ROC_AUC': [0.6332, 0.6616, auc_lr,  auc_xgb,  auc_lgb],
    'Recall':      [0.0423, 0.3979, rec_lr,  rec_xgb,  rec_lgb],
    'Precision':   [0.3172, 0.1990, prec_lr, prec_xgb, prec_lgb],
    'F1':          [0.0746, 0.2653, f1_lr,   f1_xgb,   f1_lgb],
    'Encoding':    ['Label', 'Label', 'One-Hot+Target',
                    'One-Hot+Target', 'One-Hot+Target']
}).round(4)

print("=" * 85)
print("COMPLETE MODEL COMPARISON — ALL PROFESSOR FIXES APPLIED")
print("=" * 85)
print(results.to_string(index=False))
print("=" * 85)

best_val = results.loc[results['Val_ROC_AUC'].idxmax()]
best_rec = results.loc[results['Recall'].idxmax()]

print(f"\nBest Validation ROC-AUC: {best_val['Model']}")
print(f"   Val ROC-AUC: {best_val['Val_ROC_AUC']:.4f}")
print(f"\nBest Recall: {best_rec['Model']}")
print(f"   Recall: {best_rec['Recall']:.4f}")

results.to_csv('../dashboard/final_model_comparison.csv', index=False)
print("\nFinal comparison saved!")

COMPLETE MODEL COMPARISON — ALL PROFESSOR FIXES APPLIED
                                Model      CV_ROC_AUC  Val_ROC_AUC  Recall  Precision     F1       Encoding
            XGBoost original (Week 5)             N/A       0.6332  0.0423     0.3172 0.0746          Label
      XGBoost + interactions (Week 6)             N/A       0.6616  0.3979     0.1990 0.2653          Label
                     LR + PCA (Fix 2) 0.9208 ± 0.0014       0.5516  0.4514     0.1344 0.2072 One-Hot+Target
 XGBoost no PCA + One-Hot (Fix 1+2+3) 0.9550 ± 0.0011       0.6143  0.6313     0.1467 0.2380 One-Hot+Target
LightGBM no PCA + One-Hot (Fix 1+2+3) 0.9544 ± 0.0017       0.6270  0.5826     0.1546 0.2444 One-Hot+Target

Best Validation ROC-AUC: XGBoost + interactions (Week 6)
   Val ROC-AUC: 0.6616

Best Recall: XGBoost no PCA + One-Hot (Fix 1+2+3)
   Recall: 0.6313

Final comparison saved!
